In [1]:
# -*- coding: utf-8 -*-
"""
개체 단위 랜덤 층화 분할 (시간 무관, DuckDB 중심)
- train_raw : val_tune_raw : val_calib_raw : test_raw = 6 : 1 : 1 : 2
- serial_number의 _n 꼬리표는 같은 물리 개체로 통합
- 개체 누수 방지: 같은 base serial은 반드시 한 split에만 배정
"""

import json
import os
import traceback
from pathlib import Path

import duckdb


# =========================
# 설정
# =========================
PARQUET_PATH = r"C:\Workspace\06_ML_projdect\26_1_COIN\data\ST4000DM000_v3.parquet"
RANDOM_SEED = 42
SAVE_SPLIT_PARQUET = True
PARQUET_COMPRESSION = "ZSTD"  # 저장공간 절약 우선
OUT_DIR = Path(r"C:\Workspace\06_ML_projdect\26_1_COIN\data\split_group_stratified")
DUCKDB_THREADS = 4
DUCKDB_MEMORY_LIMIT = "8GB"
DUCKDB_TEMP_DIR = OUT_DIR / "_duckdb_tmp"
CHECKPOINT_PATH = OUT_DIR / "_split_checkpoint.json"


def build_entity_split_table(con: duckdb.DuckDBPyConnection) -> None:
    # 개체 라벨(고장 여부) 생성
    con.execute(
        """
        CREATE OR REPLACE TEMP TABLE entity_label AS
        SELECT
            regexp_replace(serial_number, '_[0-9]+$', '') AS entity_id,
            CAST(MAX(failure) AS BIGINT) AS entity_failed
        FROM read_parquet(?)
        GROUP BY 1
        """,
        [PARQUET_PATH],
    )

    # strata(entity_failed)별로 해시 기반 셔플 순번 생성 (시드 고정)
    con.execute(
        """
        CREATE OR REPLACE TEMP TABLE entity_ranked AS
        SELECT
            entity_id,
            entity_failed,
            ROW_NUMBER() OVER (
                PARTITION BY entity_failed
                ORDER BY hash(entity_id || '|' || CAST(? AS VARCHAR))
            ) AS rn,
            COUNT(*) OVER (PARTITION BY entity_failed) AS grp_n
        FROM entity_label
        """,
        [RANDOM_SEED],
    )

    # 6:1:1:2 비율 할당 (층화 유지)
    # floor 기준 절단 후 remainder는 test_raw로 들어감
    con.execute(
        """
        CREATE OR REPLACE TEMP TABLE entity_split AS
        WITH c AS (
            SELECT
                entity_id,
                entity_failed,
                rn,
                grp_n,
                CAST(FLOOR(grp_n * 0.6) AS BIGINT) AS c1,
                CAST(FLOOR(grp_n * 0.1) AS BIGINT) AS c2,
                CAST(FLOOR(grp_n * 0.1) AS BIGINT) AS c3
            FROM entity_ranked
        )
        SELECT
            entity_id,
            entity_failed,
            CASE
                WHEN rn <= c1 THEN 'train_raw'
                WHEN rn <= c1 + c2 THEN 'val_tune_raw'
                WHEN rn <= c1 + c2 + c3 THEN 'val_calib_raw'
                ELSE 'test_raw'
            END AS split
        FROM c
        """
    )


def print_summary(con: duckdb.DuckDBPyConnection) -> None:
    entity_summary = con.execute(
        """
        SELECT
            split,
            COUNT(*) AS total_entities,
            SUM(entity_failed) AS failed_entities,
            ROUND(SUM(entity_failed) * 1.0 / COUNT(*), 6) AS failed_ratio
        FROM entity_split
        GROUP BY split
        ORDER BY split
        """
    ).fetchdf()

    row_summary = con.execute(
        """
        SELECT
            e.split,
            COUNT(*) AS total_rows,
            SUM(CAST(d.failure AS BIGINT)) AS failed_rows,
            ROUND(SUM(CAST(d.failure AS DOUBLE)) / COUNT(*), 6) AS failed_row_ratio
        FROM read_parquet(?) d
        JOIN entity_split e
          ON regexp_replace(d.serial_number, '_[0-9]+$', '') = e.entity_id
        GROUP BY e.split
        ORDER BY e.split
        """,
        [PARQUET_PATH],
    ).fetchdf()

    leak_check = con.execute(
        """
        SELECT MAX(split_cnt) FROM (
            SELECT entity_id, COUNT(DISTINCT split) AS split_cnt
            FROM entity_split
            GROUP BY entity_id
        )
        """
    ).fetchone()[0]

    print("\n[개체 기준 분할 요약]")
    print(entity_summary.to_string(index=False))

    print("\n[row 기준 분할 요약(참고)]")
    print(row_summary.to_string(index=False))

    print("\n[누수 체크]")
    print(f"한 개체가 속한 split 최대 개수: {leak_check} (정상은 1)")


def save_split_files(con: duckdb.DuckDBPyConnection, out_dir: Path) -> None:
    out_dir.mkdir(parents=True, exist_ok=True)

    split_list = [r[0] for r in con.execute("SELECT DISTINCT split FROM entity_split ORDER BY split").fetchall()]
    total = len(split_list)

    expected_map = {
        row[0]: int(row[1])
        for row in con.execute(
            """
            SELECT e.split, COUNT(*) AS expected_rows
            FROM read_parquet(?) d
            JOIN entity_split e
              ON regexp_replace(d.serial_number, '_[0-9]+$', '') = e.entity_id
            GROUP BY e.split
            """,
            [PARQUET_PATH],
        ).fetchall()
    }

    checkpoint = {"completed": []}
    if CHECKPOINT_PATH.exists():
        try:
            checkpoint = json.loads(CHECKPOINT_PATH.read_text(encoding="utf-8"))
        except Exception:
            checkpoint = {"completed": []}

    for idx, sp in enumerate(split_list, start=1):
        out_path = out_dir / f"{sp}.parquet"
        tmp_path = out_dir / f".{sp}.tmp.parquet"
        safe_tmp_path = str(tmp_path).replace("'", "''")
        safe_split = str(sp).replace("'", "''")
        expected_rows = int(expected_map.get(sp, 0))

        if sp in checkpoint.get("completed", []) and out_path.exists():
            existing_rows = con.execute("SELECT COUNT(*) FROM read_parquet(?)", [str(out_path)]).fetchone()[0]
            if existing_rows == expected_rows and expected_rows > 0:
                print(
                    f"[{idx}/{total}] 건너뜀(체크포인트 유효): {out_path} "
                    f"(rows={existing_rows:,}, size={out_path.stat().st_size:,} bytes)"
                )
                continue

        if out_path.exists():
            existing_rows = con.execute("SELECT COUNT(*) FROM read_parquet(?)", [str(out_path)]).fetchone()[0]
            if existing_rows == expected_rows and expected_rows > 0:
                print(
                    f"[{idx}/{total}] 건너뜀(기존 유효 파일): {out_path} "
                    f"(rows={existing_rows:,}, size={out_path.stat().st_size:,} bytes)"
                )
                checkpoint.setdefault("completed", []).append(sp)
                CHECKPOINT_PATH.write_text(json.dumps(checkpoint, ensure_ascii=False, indent=2), encoding="utf-8")
                continue
            out_path.unlink()

        if tmp_path.exists():
            tmp_path.unlink()

        print(f"[{idx}/{total}] 저장 중: {sp} (expected_rows={expected_rows:,}) ...")

        try:
            con.execute(
                f"""
                COPY (
                    SELECT d.*
                    FROM read_parquet('{PARQUET_PATH}') d
                    JOIN entity_split e
                      ON regexp_replace(d.serial_number, '_[0-9]+$', '') = e.entity_id
                    WHERE e.split = '{safe_split}'
                ) TO '{safe_tmp_path}' (FORMAT PARQUET, COMPRESSION {PARQUET_COMPRESSION})
                """
            )

            actual_rows = con.execute("SELECT COUNT(*) FROM read_parquet(?)", [str(tmp_path)]).fetchone()[0]
            if expected_rows == 0 or actual_rows == 0 or expected_rows != actual_rows:
                raise RuntimeError(
                    f"저장 검증 실패: {tmp_path} | expected={expected_rows:,}, actual={actual_rows:,}"
                )

            os.replace(tmp_path, out_path)
            checkpoint.setdefault("completed", []).append(sp)
            checkpoint["completed"] = sorted(set(checkpoint["completed"]))
            CHECKPOINT_PATH.write_text(json.dumps(checkpoint, ensure_ascii=False, indent=2), encoding="utf-8")

            print(
                f"저장 완료: {out_path} "
                f"(rows={actual_rows:,}, size={out_path.stat().st_size:,} bytes)"
            )
        except Exception:
            if tmp_path.exists():
                tmp_path.unlink()
            raise


def validate_saved_splits(con: duckdb.DuckDBPyConnection, out_dir: Path) -> None:
    expected_files = [f"{r[0]}.parquet" for r in con.execute("SELECT DISTINCT split FROM entity_split ORDER BY split").fetchall()]
    existing_files = sorted([p.name for p in out_dir.glob("*.parquet")])
    if expected_files != existing_files:
        raise RuntimeError(f"파일 셋 불일치: expected={expected_files}, existing={existing_files}")

    src_rows, src_fail_rows, src_entities = con.execute(
        """
        SELECT
            COUNT(*) AS total_rows,
            SUM(CAST(failure AS BIGINT)) AS fail_rows,
            COUNT(DISTINCT regexp_replace(serial_number, '_[0-9]+$', '')) AS entities
        FROM read_parquet(?)
        """,
        [PARQUET_PATH],
    ).fetchone()

    split_glob = str(out_dir / "*.parquet")
    split_rows, split_fail_rows, split_entities = con.execute(
        """
        SELECT
            COUNT(*) AS total_rows,
            SUM(CAST(failure AS BIGINT)) AS fail_rows,
            COUNT(DISTINCT regexp_replace(serial_number, '_[0-9]+$', '')) AS entities
        FROM read_parquet(?)
        """,
        [split_glob],
    ).fetchone()

    if src_rows != split_rows or src_fail_rows != split_fail_rows or src_entities != split_entities:
        raise RuntimeError(
            "원본-분할 무결성 불일치: "
            f"rows {src_rows:,}!={split_rows:,}, "
            f"failed_rows {src_fail_rows:,}!={split_fail_rows:,}, "
            f"entities {src_entities:,}!={split_entities:,}"
        )

    con.execute("DROP TABLE IF EXISTS split_entities")
    for i, name in enumerate(existing_files):
        file_path = str(out_dir / name).replace("'", "''")
        split_name = name.replace(".parquet", "").replace("'", "''")
        if i == 0:
            con.execute(
                f"""
                CREATE TEMP TABLE split_entities AS
                SELECT '{split_name}' AS split,
                       regexp_replace(serial_number, '_[0-9]+$', '') AS entity_id
                FROM read_parquet('{file_path}')
                GROUP BY 1, 2
                """
            )
        else:
            con.execute(
                f"""
                INSERT INTO split_entities
                SELECT '{split_name}' AS split,
                       regexp_replace(serial_number, '_[0-9]+$', '') AS entity_id
                FROM read_parquet('{file_path}')
                GROUP BY 1, 2
                """
            )

    dup_entities = con.execute(
        """
        SELECT COUNT(*)
        FROM (
            SELECT entity_id
            FROM split_entities
            GROUP BY entity_id
            HAVING COUNT(DISTINCT split) > 1
        )
        """
    ).fetchone()[0]
    if dup_entities != 0:
        raise RuntimeError(f"개체 중복 배정 감지: {dup_entities:,} entities")

    src_schema = con.execute("DESCRIBE SELECT * FROM read_parquet(?)", [PARQUET_PATH]).fetchdf()
    src_cols = src_schema["column_name"].tolist()
    bad_schema_files = []
    for name in existing_files:
        fp = str(out_dir / name)
        file_schema = con.execute("DESCRIBE SELECT * FROM read_parquet(?)", [fp]).fetchdf()
        file_cols = file_schema["column_name"].tolist()
        if file_cols != src_cols:
            bad_schema_files.append(name)

    if bad_schema_files:
        raise RuntimeError(f"스키마 불일치 파일: {bad_schema_files}")

    print("\n[저장 무결성 검증 통과]")
    print(f"- 파일 셋: {existing_files}")
    print(f"- rows: {src_rows:,}")
    print(f"- failed_rows: {src_fail_rows:,}")
    print(f"- entities: {src_entities:,}")
    print("- 개체 중복 배정: 0")
    print("- 스키마: 원본과 동일")


def main():
    con = duckdb.connect()
    try:
        DUCKDB_TEMP_DIR.mkdir(parents=True, exist_ok=True)
        safe_temp_dir = str(DUCKDB_TEMP_DIR).replace("'", "''")
        con.execute(f"PRAGMA threads={DUCKDB_THREADS}")
        con.execute(f"PRAGMA memory_limit='{DUCKDB_MEMORY_LIMIT}'")
        con.execute(f"PRAGMA temp_directory='{safe_temp_dir}'")

        print("[설정]")
        print(f"- random_seed: {RANDOM_SEED}")
        print(f"- parquet_compression: {PARQUET_COMPRESSION}")
        print(f"- duckdb_threads: {DUCKDB_THREADS}")
        print(f"- duckdb_memory_limit: {DUCKDB_MEMORY_LIMIT}")
        print(f"- duckdb_temp_dir: {DUCKDB_TEMP_DIR}")
        print("- split_ratio: train_raw:val_tune_raw:val_calib_raw:test_raw = 6:1:1:2")
        print("- 엔진: DuckDB 중심 (pandas/numpy 미사용)")

        print("\n[1/4] entity_split 생성 중...")
        build_entity_split_table(con)

        print("[2/4] 분할 요약 계산 중...")
        print_summary(con)

        if SAVE_SPLIT_PARQUET:
            print("[3/4] split 저장 중...")
            save_split_files(con, OUT_DIR)
            print("[4/4] 저장 무결성 검증 중...")
            validate_saved_splits(con, OUT_DIR)

    except Exception as e:
        print("\n[실패] 실행 중 예외가 발생했습니다.")
        print(f"- error_type: {type(e).__name__}")
        print(f"- error_message: {e}")
        print("- traceback:")
        print(traceback.format_exc())
        raise
    finally:
        con.close()


if __name__ == "__main__":
    main()

[설정]
- random_seed: 42
- parquet_compression: ZSTD
- duckdb_threads: 4
- duckdb_memory_limit: 8GB
- duckdb_temp_dir: C:\Workspace\06_ML_projdect\26_1_COIN\data\split_group_stratified\_duckdb_tmp
- split_ratio: train_raw:val_tune_raw:val_calib_raw:test_raw = 6:1:1:2
- 엔진: DuckDB 중심 (pandas/numpy 미사용)

[1/4] entity_split 생성 중...
[2/4] 분할 요약 계산 중...

[개체 기준 분할 요약]
        split  total_entities  failed_entities  failed_ratio
     test_raw            7391           1135.0      0.153565
    train_raw           22161           3400.0      0.153423
val_calib_raw            3692            566.0      0.153304
 val_tune_raw            3692            566.0      0.153304

[row 기준 분할 요약(참고)]
        split  total_rows  failed_rows  failed_row_ratio
     test_raw    15898947      11232.0          0.000706
    train_raw    47858472      33573.0          0.000702
val_calib_raw     7932346       5568.0          0.000702
 val_tune_raw     8004395       5592.0          0.000699

[누수 체크]
한 개체가 속한 split 최대